<img src="logo.png" alt="Vegeta" width="240">

# Delta-wing medical delivery drone — two sizes, four powerplants, a parachute landing

A fast outdoor courier for medicine: a **delta flying wing** (a triangle with a central body, twin fins and a nose
payload bay) that is catapult-launched, cruises to the site, cuts its motor and **lands whole under a timed
parachute**. Two sizes from one parametric design (**2 m** and **3 m** span), four ways to push it — a **pusher
propeller** or a **jet**, each **electric** or on **liquid fuel** — and one question at the end: **does the
medicine arrive intact?**

Propulsion is approximated, never resolved: the propeller is a Boreas blade-element model and, in the CFD, a
**rotor disk**; the jet (an electric ducted fan or a micro turbojet) is a thrust curve and, in the CFD, a
**prescribed-thrust disk**. Liquid fuel burns along the mission, so the aircraft gets lighter and its drag and
consumption fall; the CFD is run once, at the fixed mid-mission mass, for simplicity.

```
1  the two sizes (Dedalus): geometry, mass budgets for the eight cases
2  aerodynamics: delta-wing lift (Polhamus) and polar; cruise, stall, launch; whole-aircraft RANS (Aeromant)
3  propulsion: {pusher, jet} x {electric, liquid}: thrust vs drag, consumption along the mission, range, noise
4  in-flight CFD with the propulsion: rotor disk (pusher) or thrust disk (jet) behind the body
5  structure (Talos): wing pull-up and launch, fin gust, modes vs the propeller lines
6  landing: the timed parachute — opening shock, descent in wind, touchdown, landing scatter
7  does the medicine survive? cruise vibration, opening shock and touchdown at the vials, for all eight cases
8  export and the descent movie
```
The OpenFOAM cells run when you run them (`VEGETA_SKIP_OPENFOAM=1` skips them; the polar then stands in).
Every number that is an assumption says so.

In [ ]:
import json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, aeromant, boreas, chronos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.boreas.range import Aircraft, ElectricFan, ElectricProp, Leg, PistonProp, Turbojet, fly_mission, max_range
from vegeta.chronos.parachute import Body, Parachute, Wind, crush_pulse, landing_scatter, simulate_drop
import sys; sys.path.insert(0, "designs")
from delta_wing import SIZE_2M, SIZE_3M

ROOT = Path("_runs/delta_wing"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
MOVIES = Path("output") / "16_delta_wing_delivery"; MOVIES.mkdir(parents=True, exist_ok=True)   # every movie of this notebook
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
RHO, G = 1.225, 9.81

# -- the mission and its requirements (engineering inputs) --
RADIUS_KM = 20.0                    # one-way delivery: the drone lands at the site
WIND = 8.0                          # m/s: the outdoor wind it must handle
LANDING_CIRCLE_M = 30.0             # radius the drone must land within (around the predicted point)
CRUISE_ALT_M = 120.0                # release altitude for the parachute
G_VIALS = 50.0                      # assumed: glass vials in a padded cold box survive short pulses below this
G_VACCINE = 15.0                    # assumed: a sustained/repeated shock limit for the contents
G_VIB_RMS = 1.0                     # assumed: continuous vibration limit at the box (rms g) in cruise
SIZES = {"2 m": dict(SIZE_2M, payload_kg=2.0, v_cruise=35.0), "3 m": dict(SIZE_3M, payload_kg=5.0, v_cruise=40.0)}
design = dedalus.load_design("designs/delta_wing.py:DeltaWing")
PARAM_NAMES = {q.name for q in design.params}
pd.DataFrame(design.params.table()).set_index("name").head(12)

## 1. The two sizes (Dedalus)

One parametric delta: span, root chord, body, fins, nose bay; `propulsion` swaps the tail end (a spinner boss
for the pusher, a nozzle duct for the jet). The airframe mass comes from the CAD: a foam-core wing with a
glass-fibre skin (areal and volumetric densities are **assumptions**), plus systems, the parachute, the powerplant
and its energy store. Eight cases = two sizes x four powerplants.

In [ ]:
geo = {}
for name, s in SIZES.items():
    p = design.resolve(**{k: v for k, v in s.items() if k in PARAM_NAMES})
    geo[name] = dict(params=p, aircraft=design.generate(part="aircraft", propulsion="pusher", **{k: v for k, v in s.items() if k in PARAM_NAMES}))
    m = geo[name]["aircraft"].measure()
    geo[name].update(S=s["span"] / 1000 * (p["root_chord"] + p["tip_chord"]) / 2 / 1000, area_m2=m["surface_area"] / 1e6,
                     wing_volume_m3=design.generate(part="wing", **{k: v for k, v in s.items() if k in PARAM_NAMES}).volume / 1e9)
    geo[name]["AR"] = (s["span"] / 1000) ** 2 / geo[name]["S"]
    geo[name]["sweep_deg"] = math.degrees(math.atan2(p["root_chord"] - p["tip_chord"], s["span"] / 2))
    print(f"{name}: planform {geo[name]['S']:.2f} m^2, AR {geo[name]['AR']:.2f}, LE sweep {geo[name]['sweep_deg']:.0f} deg, "
          f"wetted {geo[name]['area_m2']:.2f} m^2, wing volume {geo[name]['wing_volume_m3'] * 1000:.0f} L")
dviz.show(dviz.plot3d(geo["2 m"]["aircraft"], dedalus.Geometry.from_cadquery(design.generate(propulsion="jet", **{k: v for k, v in SIZES["3 m"].items() if k in PARAM_NAMES}).shape.translate((0, 0, 600)), name="3 m jet (600 mm above)")))

In [ ]:
SKIN_KG_M2, CORE_KG_M3 = 0.9, 30.0             # assumed: 2 x GFRP + paint over the wetted area; XPS foam core
POWERPLANTS = {                                   # dry masses [kg] per size (assumed, datasheet-class values)
    "pusher / electric": {"2 m": 0.7, "3 m": 1.4},  "pusher / petrol": {"2 m": 1.6, "3 m": 3.2},
    "jet / electric":    {"2 m": 0.9, "3 m": 1.8},  "jet / turbojet":  {"2 m": 1.5, "3 m": 2.6},
}
STORE = {"pusher / electric": {"2 m": 1.6, "3 m": 4.0}, "jet / electric": {"2 m": 1.6, "3 m": 4.0},     # battery mass
         "pusher / petrol": {"2 m": 1.2, "3 m": 3.0}, "jet / turbojet": {"2 m": 1.2, "3 m": 3.0}}       # fuel mass at take-off
SYSTEMS = {"2 m": 0.9, "3 m": 1.4}; CHUTE = {"2 m": 0.5, "3 m": 1.0}
rows = {}
for name, g in geo.items():
    airframe = g["area_m2"] * SKIN_KG_M2 + g["wing_volume_m3"] * CORE_KG_M3
    for case in POWERPLANTS:
        rows[(name, case)] = {"airframe": airframe, "systems": SYSTEMS[name], "parachute": CHUTE[name], "payload": SIZES[name]["payload_kg"],
                              "powerplant": POWERPLANTS[case][name], "energy store": STORE[case][name]}
mass = pd.DataFrame(rows).T
mass["take-off"] = mass.sum(axis=1)
mass["payload fraction"] = mass["payload"] / mass["take-off"]
mass.round(2)

## 2. Aerodynamics

A delta wing lifts with its leading-edge vortices: the **Polhamus** suction analogy, `CL = Kp sin α cos²α +
Kv cos α sin²α`, with `Kp` from the aspect ratio and sweep and `Kv = π`. Its induced drag is `CL tan α` (the
leading-edge suction is lost), so the polar's `k ≈ 1/Kp` — higher than an elliptic wing's: a **conservative**
polar (a rounded leading edge keeps part of the suction and gains 20–30 % in L/D). `cd0` is an **assumption** for a
clean flying wing with a body and fins. Cruise, stall and the catapult speed follow; the
whole-aircraft RANS (Aeromant, with no propulsion) checks the polar at the cruise angle.

In [ ]:
CD0, CL_MAX = 0.022, 0.9                          # assumed: clean flying wing + body + fins; vortex lift stalls late
def polhamus(AR, sweep_deg):
    kp = 2 * math.pi * AR / (2 + math.sqrt(4 + AR**2 * (1 + math.tan(math.radians(sweep_deg)) ** 2)))
    return kp, math.pi
aero = {}
for name, g in geo.items():
    kp, kv = polhamus(g["AR"], g["sweep_deg"])
    a = np.radians(np.linspace(0, 30, 61))
    cl = kp * np.sin(a) * np.cos(a) ** 2 + kv * np.cos(a) * np.sin(a) ** 2
    cd = CD0 + cl * np.tan(a)
    aero[name] = dict(Kp=kp, Kv=kv, k=1 / kp, alpha=a, cl=cl, cd=cd)
    print(f"{name}: Kp {kp:.2f}/rad, k = 1/Kp = {1 / kp:.3f}, L/D max {np.max(cl[1:] / cd[1:]):.1f} at {np.degrees(a[1 + np.argmax(cl[1:] / cd[1:])]):.0f} deg")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, ar in aero.items():
    ax[0].plot(np.degrees(ar["alpha"]), ar["cl"], label=name); ax[1].plot(ar["cd"], ar["cl"], label=name)
ax[0].set(xlabel="angle of attack [deg]", ylabel="CL", title="Polhamus delta-wing lift"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].set(xlabel="CD", ylabel="CL", title="polar"); ax[1].grid(alpha=0.3); fig.tight_layout()

In [ ]:
def aircraft_of(name, case):
    m = mass.loc[(name, case)]
    return Aircraft(f"{name} {case}", float(m["airframe"] + m["systems"] + m["parachute"] + m["powerplant"]), float(m["payload"]),
                    geo[name]["S"], CD0, aero[name]["k"], CL_MAX, RHO)
flight = {}
for name in SIZES:
    ac = aircraft_of(name, "pusher / electric"); m_to = float(mass.loc[(name, "pusher / electric"), "take-off"])
    v = SIZES[name]["v_cruise"]; d, cl = ac.drag(v, m_to)
    alpha_cruise = math.degrees(cl / aero[name]["Kp"])
    flight[name] = dict(v=v, drag_N=d, cl=cl, alpha_deg=alpha_cruise, v_stall=ac.stall_speed(m_to), v_launch=1.3 * ac.stall_speed(m_to))
pd.DataFrame(flight).T.round(2)

In [ ]:
# whole-aircraft RANS at the cruise angle (no propulsion): the polar's check. Flow +x, so the aircraft is turned nose-upstream.
def flow_frame(name, propulsion):
    s = {k: v for k, v in SIZES[name].items() if k in PARAM_NAMES}
    g = design.generate(part="aircraft", propulsion=propulsion, angle_of_attack_deg=-flight[name]["alpha_deg"], **s)   # nose up in the +X frame
    return dedalus.Geometry.from_cadquery(g.shape.rotate((0, 0, 0), (0, 0, 1), 180), name=f"{name} {propulsion} nose upstream")
cfd_clean = {}
for name in SIZES:
    g = flow_frame(name, "pusher"); stl = g.export_stl(ROOT / "cad" / f"{name.replace(' ', '')}_clean.stl", tolerance=0.3)
    c = geo[name]["params"]["root_chord"] / 1000
    case = aeromant.CFDCase("rans_ksst_external", stl, dict(velocity=flight[name]["v"], kinematic_viscosity=1.5e-5, density=RHO,
                            reference_area=geo[name]["S"], reference_length=0.5 * c, center_of_rotation=(-0.45 * c, 0.0, 0.0),
                            iterations=400, residual_target=1e-4, cells_per_length=2.0, surface_level=4, near_level=3, wake_level=2),
                            workdir=ROOT / "cfd_clean" / name.replace(" ", ""), geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    print(name, case.prepare(overwrite=True).status, "->", case.workdir)
    cfd_clean[name] = (case, case.run(progress=True) if RUN_CFD else None)
if not RUN_CFD:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): the cases are prepared; run them on a machine with OpenFOAM")
else:
    for name, (case, r) in cfd_clean.items():
        if r.ok:
            m = r.metrics; print(f"{name}: Cl {m['Cl']:.3f} (polar {flight[name]['cl']:.3f}), Cd {m['Cd']:.4f} (polar {CD0 + aero[name]['k'] * flight[name]['cl']**2:.4f}), converged {m['converged']}")
            aviz.show(aviz.plot_field_slice(case, "U", normal="y"))

## 3. Propulsion — four cases

| case | thrust from | consumption | mass along the mission |
|---|---|---|---|
| pusher / electric | Boreas BEMT propeller on a brushless motor | electrical power from the motor model, LiPo battery | constant |
| pusher / petrol | the same propeller family on a small two-stroke | shaft power x BSFC (assumed 0.5 kg/kWh) | falls as fuel burns |
| jet / electric | electric ducted fan: fan momentum theory (assumed 55 % electrical-to-jet efficiency) | electrical power, the same battery | constant |
| jet / turbojet | micro turbojet: static thrust with a lapse with speed | thrust x TSFC (assumed 0.15 kg/N/h) + idle flow | falls as fuel burns |

The mission is one way: climb to 120 m, cruise 20 km, one minute over the site, then the parachute. The mission is
flown step by step (`boreas.range.fly_mission`): lift equals the current weight at every step, so a liquid-fuel
aircraft's drag and consumption fall as it burns. Then the still-air and headwind **range** (`max_range`, 10 %
reserve), and the noise of the propeller cases (Gutin tones + broadband; fans and jets: not modelled here).

In [ ]:
AF = boreas.Airfoil(name="thin cambered blade section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-2.5, cl_max=1.1, cd0=0.02, k=0.04, source="assumed")
def prop_of(d_in, p_in):
    di, pi = boreas.inches(d_in, p_in)
    return boreas.Propeller.from_pitch(f"{d_in}x{p_in}", di, pi, blades=2, chord_root_m=0.06 * di, chord_max_m=0.09 * di, chord_tip_m=0.025 * di,
                                       mass_kg=0.03 * (di / 0.33) ** 2, rotor_mass_kg=0.1 * (di / 0.33) ** 2, notes="generic planform")
HW = {   # hardware per size (datasheet-class assumptions)
    "2 m": dict(eprop=(13, 8), motor=boreas.Motor("4130-500KV", 500, 0.04, 1.2, 80, 0.4), batt=boreas.Battery("8S 12 Ah", 8, 12.0, usable_fraction=0.8, mass_kg=1.6),
                pprop=(14, 10), engine_w=2200.0, rpm_max=11000.0, fan_d=0.090, fan_w=4000.0, jet_n=60.0),
    "3 m": dict(eprop=(20, 13), motor=boreas.Motor("6374-250KV", 250, 0.02, 1.5, 150, 0.9), batt=boreas.Battery("12S 20 Ah", 12, 20.0, usable_fraction=0.8, mass_kg=4.0),
                pprop=(20, 14), engine_w=5500.0, rpm_max=8500.0, fan_d=0.150, fan_w=12000.0, jet_n=130.0),
}
def powerplant(name, case):
    h = HW[name]
    if case == "pusher / electric":
        return ElectricProp(boreas.Propulsion(prop_of(*h["eprop"]), AF, h["motor"], h["batt"], RHO), name=case)
    if case == "pusher / petrol":
        return PistonProp(prop_of(*h["pprop"]), AF, h["engine_w"], 0.5, fuel_kg=STORE[case][name], rpm_max=h["rpm_max"], rho=RHO, name=case)
    if case == "jet / electric":
        return ElectricFan(math.pi * (h["fan_d"] / 2) ** 2, h["fan_w"], h["batt"], 0.55, RHO, name=case)
    return Turbojet(h["jet_n"], 0.15, fuel_kg=STORE[case][name], name=case)
PP = {(name, case): powerplant(name, case) for name in SIZES for case in POWERPLANTS}
AC = {(name, case): aircraft_of(name, case) for name in SIZES for case in POWERPLANTS}
for (name, case), pp in PP.items():
    assert abs(pp.store_mass() - STORE[case][name]) < 1e-9, (name, case)
avail = pd.DataFrame({(n, c): {"drag at cruise [N]": AC[(n, c)].drag(SIZES[n]["v_cruise"], float(mass.loc[(n, c), "take-off"]))[0],
                               "max thrust at cruise [N]": pp.thrust_max(SIZES[n]["v_cruise"]), "static thrust [N]": pp.thrust_max(0.0)}
                      for (n, c), pp in PP.items()}).T
avail["margin"] = avail["max thrust at cruise [N]"] / avail["drag at cruise [N]"]
avail.round(2)

In [ ]:
def mission_legs(name):
    v = SIZES[name]["v_cruise"]
    return [Leg("climb to 120 m", 0.8 * v, duration_s=CRUISE_ALT_M / 3.0, climb_m_s=3.0),
            Leg("cruise to the site", v, distance_m=RADIUS_KM * 1000),
            Leg("over the site", 0.8 * v, duration_s=60.0)]
logs, cmp = {}, {}
for (name, case), pp in tqdm(PP.items(), desc="missions"):
    ac = AC[(name, case)]; v = SIZES[name]["v_cruise"]
    log = fly_mission(ac, pp, mission_legs(name), dt=2.0)
    still = max_range(ac, pp, v, reserve=0.1, dt=5.0); head = max_range(ac, pp, v, wind_m_s=-WIND, reserve=0.1, dt=5.0)
    logs[(name, case)] = log
    unit = "Wh" if pp.kind == "electric" else "kg"
    cmp[(name, case)] = {"take-off [kg]": log.mass[0], "landing [kg]": log.mass[-1], "cruise thrust [N]": float(np.median(log.thrust)),
                         f"mission [{unit}]": float(log.energy_used[-1]), "of available [%]": 100 * log.energy_used[-1] / pp.energy_available(),
                         "mission [min]": log.endurance_min, "range still air [km]": still.range_km, f"range into {WIND:.0f} m/s [km]": head.range_km,
                         "endurance [min]": still.endurance_min, "completed": log.stopped is None}
comparison = pd.DataFrame(cmp).T
comparison.round(2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for (name, case), log in logs.items():
    ls = "-" if name == "2 m" else "--"
    ax[0].plot(log.t / 60, log.mass, ls, label=f"{name} {case}"); ax[1].plot(log.t / 60, log.thrust, ls); ax[2].plot(log.t / 60, log.drag, ls)
ax[0].set(xlabel="time [min]", ylabel="mass [kg]", title="mass along the mission (liquid fuel burns off)"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="time [min]", ylabel="thrust [N]", title="thrust needed"); ax[2].set(xlabel="time [min]", ylabel="drag [N]", title="drag (lift = weight)")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# how much energy or fuel per delivery, and how far each could go on a full store
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
comparison[["range still air [km]", f"range into {WIND:.0f} m/s [km]"]].plot.barh(ax=ax[0], title="range on a full store (10 % reserve)"); ax[0].axvline(RADIUS_KM, color="#c62828", ls="--")
comparison["of available [%]"].plot.barh(ax=ax[1], title="mission energy / fuel as % of the store", color="#546e7a"); ax[1].axvline(100, color="#c62828", ls="--")
for a in ax: a.grid(alpha=0.3, axis="x")
fig.tight_layout()

In [ ]:
# noise of the propeller cases at 1 m, 90 deg (Gutin tones + broadband); fans and turbojets are not modelled here
noise = {}
for (name, case), pp in PP.items():
    if "pusher" not in case:
        noise[(name, case)] = {"rpm": float("nan"), "BPF [Hz]": float("nan"), "total dB(A)": float("nan"), "note": "EDF / turbojet: not modelled (typically 95-110 dB at 1 m)"}
        continue
    prop = pp.propulsion.prop if case == "pusher / electric" else pp.prop
    T = float(np.median(logs[(name, case)].thrust)); v = SIZES[name]["v_cruise"]
    op = boreas.rpm_for_thrust(prop, AF, T, v, RHO, rpm_max=20000)
    tones = boreas.gutin_harmonics(prop, T, op.torque, op.rpm, 1.0, 90.0, boreas.AIR, harmonics=5)
    bb = boreas.broadband_level(prop, T, op.rpm, 1.0, boreas.AIR)
    noise[(name, case)] = {"rpm": op.rpm, "BPF [Hz]": tones["blade_pass_hz"], "total dB(A)": 10 * math.log10(10 ** (tones["total_tonal_db"] / 10) + 10 ** (bb / 10)),
                           "note": "Gutin + broadband" + (" (engine exhaust adds ~5-10 dB)" if "petrol" in case else "")}
pd.DataFrame(noise).T

## 4. In-flight CFD with the propulsion

The same aircraft in the same flow, now with its propulsion at the tail: the **pusher** as a rotor disk (blade
elements from the Boreas propeller at the rpm the mission needs), the **jet** as a prescribed-thrust disk at the
nozzle (a momentum source: the cruise thrust, no swirl). One disk behind the body, so the `hull_rotor_disk`
template. The forces are the airframe's in the propeller's inflow / the jet's entrainment; the mass is fixed at
the mid-mission value. Both cases are prepared for both sizes; they run when OpenFOAM runs.

In [ ]:
def polar_table(airfoil):
    al = np.unique(np.r_[np.arange(-180, -30, 10), np.arange(-30, 31, 1), np.arange(40, 181, 10)]).astype(float)
    cl, cd = airfoil.coefficients(np.radians(al)); return [[float(a), float(d), float(l)] for a, l, d in zip(al, cl, cd)]
def tail_point(name, x_tail_mm):
    """A point on the axis x_tail_mm behind the tail (x = 0 in the design), in the flow frame (nose upstream, cruise AoA)."""
    a = math.radians(flight[name]["alpha_deg"]); xt = -x_tail_mm / 1000
    return [-xt * math.cos(a), 0.0, xt * math.sin(a)], [-math.cos(a), 0.0, math.sin(a)]       # centre, thrust direction (forward)
cfd_prop = {}
for name in SIZES:
    c = geo[name]["params"]["root_chord"] / 1000; v = SIZES[name]["v_cruise"]
    for case, propulsion in (("pusher / electric", "pusher"), ("jet / turbojet", "jet")):
        g = flow_frame(name, propulsion); stl = g.export_stl(ROOT / "cad" / f"{name.replace(' ', '')}_{propulsion}.stl", tolerance=0.3)
        pp = PP[(name, case)]; T = float(np.median(logs[(name, case)].thrust))
        centre, axis = tail_point(name, 60.0)
        params = dict(velocity=v, kinematic_viscosity=1.5e-5, density=RHO, reference_area=geo[name]["S"], reference_length=0.5 * c,
                      center_of_rotation=(-0.45 * c, 0.0, 0.0), iterations=600, residual_target=1e-4, cells_per_length=2.0,
                      surface_level=4, near_level=3, wake_level=2, disk1_center=centre, disk_axis=axis, disk_level=5)
        if propulsion == "pusher":
            prop = pp.propulsion.prop; op = boreas.rpm_for_thrust(prop, AF, T, v, RHO, rpm_max=20000)
            params.update(disk_model="rotorDisk", diameter=prop.diameter, rpm=op.rpm, blades=prop.blades, rotation1=1,
                          blade=[[r, b, ch] for r, b, ch in zip(prop.r, prop.beta_deg, prop.chord)], polar=polar_table(AF))
        else:
            params.update(disk_model="thrust", diameter=geo[name]["params"]["nozzle_diameter"] / 1000 * 1.5, thrust=T)
        case_ = aeromant.CFDCase("hull_rotor_disk", stl, params, workdir=ROOT / "cfd_propulsion" / f"{name.replace(' ', '')}_{propulsion}",
                                 geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
        r = case_.prepare(overwrite=True); print(f"{name} {propulsion}: {r.status} ({params['disk_model']}, {T:.1f} N) -> {case_.workdir}")
        cfd_prop[(name, propulsion)] = (case_, case_.run(progress=True) if RUN_CFD else None)
if RUN_CFD:
    rows = {}
    for (name, propulsion), (case_, r) in cfd_prop.items():
        base = cfd_clean[name][1]
        if r is not None and r.ok and base is not None and base.ok:
            rows[(name, propulsion)] = {"Cl clean": base.metrics["Cl"], "Cl with propulsion": r.metrics["Cl"], "Cd clean": base.metrics["Cd"],
                                        "Cd with propulsion": r.metrics["Cd"], "drag [N]": r.metrics["drag_force_N"], "converged": r.metrics["converged"]}
            aviz.show(aviz.plot_field_slice(case_, "U", normal="y"))
    display(pd.DataFrame(rows).T.round(4))
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): four cases prepared (2 m / 3 m x rotor disk / thrust disk)")

## 5. Structure (Talos)

A foam-core, glass-skinned flying wing modelled as a **solid-equivalent** material (E, density and strength are
**assumptions** to replace with coupon tests). A half wing clamped at the symmetry plane carries: a **2.5 g pull-up**
(the lift as a pressure on the lower skin) and the **8 g catapult launch** (its own inertia). One fin takes a
side gust. The wing's modes go on a Campbell diagram against the pusher's 1P and 2P lines.

In [ ]:
FOAM = talos.Material("XPS core + GFRP skin, solid-equivalent", youngs_modulus=900.0, poissons_ratio=0.3, density=0.12e-9,
                      yield_strength=6.0, source="assumed; coupon tests needed")
N_PULL, N_LAUNCH, GUST = 2.5, 8.0, WIND
structure = {}
for name in SIZES:
    s = {k: v for k, v in SIZES[name].items() if k in PARAM_NAMES}; p = geo[name]["params"]
    c0, b2, es = p["root_chord"], p["span"] / 2, {"2 m": 28.0, "3 m": 40.0}[name]
    half = design.generate(part="half_wing", **s).export(ROOT / "fea" / name.replace(" ", "") / "half_wing", formats=("step",))
    W = float(mass.loc[(name, "pusher / electric"), "take-off"]) * G
    p_lift = N_PULL * W / (geo[name]["S"] * 1e6)                                   # MPa on the lower skin (both halves see the same pressure)
    R = [talos.SurfacesOnPlane("root", "y", 0.0, tol=0.5), talos.SurfacesInBox("lower", (-1, -1, -300, c0 + 1, b2 + 1, 0.5))]
    def model(loads, tag):
        return talos.StructuralModel(half.artifacts["step"], "mm-N-MPa", FOAM, R, [talos.FixedSupport("root")], loads, talos.MeshSettings(element_size=es), name=tag)
    base = model([talos.Pressure("lower", p_lift)], "pull_up"); mres = base.mesh(ROOT / "fea" / name.replace(" ", "") / "mesh", progress=False); mres.raise_for_status()
    def case_dir(tag):
        d = ROOT / "fea" / name.replace(" ", "") / tag
        if not d.exists(): shutil.copytree(ROOT / "fea" / name.replace(" ", "") / "mesh", d)
        return d
    pull = base.solve(case_dir("pull_up"), progress=False)
    launch = model([talos.Acceleration(ax=-N_LAUNCH * 9810.0)], "launch").solve(case_dir("launch"), progress=False)
    modes = model([], "modal").solve_modes(case_dir("modal"), n_modes=6, progress=False) if False else base.solve_modes(case_dir("modal"), n_modes=6, progress=False)
    # one fin: side gust as a pressure on one face
    fin = design.generate(part="fin", **s).export(ROOT / "fea" / name.replace(" ", "") / "fin", formats=("step",))
    z0, t, cf, hf = -0.02 * c0, p["fin_thickness"], p["fin_chord"], p["fin_height"]
    v = SIZES[name]["v_cruise"]; a_fin = 2.5                                        # 1/rad, low-aspect plate (assumed)
    p_gust = 0.5 * RHO * v * v * a_fin * (GUST / v) / 1e6                           # MPa over the fin face
    FR = [talos.SurfacesOnPlane("root", "z", z0, tol=0.5), talos.SurfacesInBox("side", (-1, t / 2 - 0.2, z0 - 1, cf + 1, t / 2 + 0.2, z0 + hf + 1))]
    fin_model = talos.StructuralModel(fin.artifacts["step"], "mm-N-MPa", FOAM, FR, [talos.FixedSupport("root")], [talos.Pressure("side", p_gust)],
                                      talos.MeshSettings(element_size=8.0), name="fin_gust")
    fin_model.mesh(ROOT / "fea" / name.replace(" ", "") / "fin_mesh", progress=False).raise_for_status()
    fin_res = fin_model.solve(ROOT / "fea" / name.replace(" ", "") / "fin_mesh", progress=False)
    structure[name] = dict(pull=pull, launch=launch, modes=modes, fin=fin_res, p_lift=p_lift, p_gust=p_gust, mesh=mres.metrics)
    print(f"{name}: mesh {mres.metrics['n_elements']} elements | pull-up {N_PULL} g: max {pull.metrics['max_von_mises']:.2f} MPa, tip {pull.metrics['max_displacement']:.1f} mm, "
          f"SF {FOAM.yield_strength / pull.metrics['max_von_mises']:.1f} | launch {N_LAUNCH} g: {launch.metrics['max_von_mises']:.2f} MPa | "
          f"fin gust {GUST:.0f} m/s: {fin_res.metrics['max_von_mises']:.2f} MPa | modes {[round(f, 1) for f in modes.metrics['frequencies_hz'][:4]]} Hz")
tviz.show(tviz.plot_results(structure["2 m"]["pull"], field="von_mises"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for i, name in enumerate(SIZES):
    st = chronos.Structure(tuple(structure[name]["modes"].metrics["frequencies_hz"]), damping_ratio=0.03)
    rpm_cruise = noise[(name, "pusher / electric")]["rpm"]
    st.campbell({"1P": 1, "2P": 2}, np.linspace(2000, 16000, 30), ax=ax[i], operating_rpm={"cruise": rpm_cruise, "full": HW[name]["motor"].kv_rpm_per_volt * HW[name]["batt"].voltage * 0.85})
    ax[i].set_title(f"{name}: wing modes vs the pusher lines")
fig.tight_layout()

## 6. Landing under the timed parachute

Over the site the motor stops; a **timer** fires the canopy; the drone hangs under it and drifts with the wind.
`chronos.parachute.simulate_drop` follows the point mass: the fall, the canopy filling over ~1.2 s (the **opening
shock** is the peak drag during inflation), the steady descent in wind with gusts, and the **touchdown** stopped by
a crush pad under the payload bay (a half-sine pulse over the pad's stroke). The canopy is sized for a
5 m/s descent at the landing mass of the heaviest case. The timer sweep shows why an unpowered delta cut at cruise
speed wants the **earliest** opening: it decelerates slowly and the fall only makes it faster.

In [ ]:
DESCENT, FILL, STROKE = 5.0, 1.2, 0.06                     # m/s target descent; canopy fill time; crush pad stroke [m] (assumed)
landing = {}
for name in SIZES:
    m_land = max(float(logs[(name, c)].mass[-1]) for c in POWERPLANTS)              # the heaviest landing (an electric case)
    chute = Parachute(Parachute(0.0).area_for_descent(m_land, DESCENT, RHO), cd=0.75, fill_time_s=FILL, mass_kg=CHUTE[name])
    body = Body(m_land - CHUTE[name], cd_a_m2=0.35 * geo[name]["S"] * 0.15)          # a tumbling delta: ~15 % of its planform (assumed)
    v = SIZES[name]["v_cruise"]
    timers = np.arange(0.5, 6.01, 0.5)
    sweep = [simulate_drop(body, chute, altitude_m=CRUISE_ALT_M, speed_m_s=0.8 * v, timer_s=t, wind=Wind(WIND), crush_stroke_m=STROKE) for t in timers]
    timer = float(timers[np.argmin([r.opening_g if r.opening_altitude_m > 40 else 1e9 for r in sweep])])   # the gentlest opening that stays above 40 m
    drop = simulate_drop(body, chute, altitude_m=CRUISE_ALT_M, speed_m_s=0.8 * v, timer_s=timer, wind=Wind(WIND, gust_rms_m_s=1.5, vertical_rms_m_s=0.5), crush_stroke_m=STROKE, seed=1)
    landing[name] = dict(chute=chute, body=body, timer=timer, drop=drop, sweep=list(zip(timers, sweep)), m_land=m_land)
    print(f"{name}: landing mass {m_land:.1f} kg, canopy {chute.area_m2:.1f} m^2 (d {math.sqrt(4 * chute.area_m2 / math.pi):.1f} m), timer {timer:.1f} s -> "
          f"opens at {drop.opening_altitude_m:.0f} m with {drop.opening_g:.1f} g, descends at {drop.descent_rate_m_s:.1f} m/s, touches down at "
          f"{drop.touchdown_vertical_m_s:.1f} m/s -> {drop.touchdown_g:.0f} g on a {STROKE * 1000:.0f} mm pad ({crush_pulse(drop.touchdown_vertical_m_s, 0.005)[2]:.0f} g without), "
          f"drift {drop.drift_m:.0f} m in {WIND:.0f} m/s wind, {drop.duration_s:.0f} s")
    print(f"      descent speed: {drop.descent_speed[np.searchsorted(drop.t, timer)]:.1f} m/s when the timer fires, {drop.descent_speed[np.searchsorted(drop.t, timer + FILL)]:.1f} m/s "
          f"with the canopy full, {drop.descent_rate_m_s:.1f} m/s steady, {drop.touchdown_vertical_m_s:.1f} m/s at touchdown (terminal {chute.terminal_speed(m_land - CHUTE[name], RHO):.1f} m/s)")
fig, ax = plt.subplots(1, 4, figsize=(19, 3.8))
for name, L in landing.items():
    d = L["drop"]; ax[0].plot(d.x, d.z, label=name); ax[1].plot(d.t, d.g, label=name)
    ln, = ax[3].plot(d.t, d.descent_speed, label=f"{name} descent speed"); ax[3].plot(d.t, d.speed, ":", color=ln.get_color(), label=f"{name} speed over the ground")
    ax[3].axhline(L["chute"].terminal_speed(L["body"].mass_kg, RHO), ls="--", lw=0.8, color=ln.get_color())
    ax[2].plot([t for t, _ in L["sweep"]], [r.opening_g for _, r in L["sweep"]], "o-", label=f"{name} opening g"); ax[2].plot([t for t, _ in L["sweep"]], [r.opening_altitude_m / 20 for _, r in L["sweep"]], "s--", label=f"{name} altitude / 20 m")
ax[0].set(xlabel="drift with the wind [m]", ylabel="altitude [m]", title="trajectory (gusty 8 m/s wind)"); ax[1].set(xlabel="time [s]", ylabel="g felt", title="opening shock, then descent")
ax[2].set(xlabel="timer [s]", title="the timer: opening shock vs altitude"); ax[3].set(xlabel="time [s]", ylabel="m/s", title="speed versus time (dashed: terminal speed)")
[a.grid(alpha=0.3) or a.legend(fontsize=8) for a in ax]; fig.tight_layout()

In [ ]:
# landing scatter: the wind is known to +-1.5 m/s, gusts 1.5 m/s rms; the drone is released upwind by the mean drift
# the release altitude is the lever on the scatter: the wind uncertainty acts for the whole descent
RELEASE_ALTS = [60.0, 90.0, 120.0]
scatter_rows, scatter = {}, {}
for name, L in landing.items():
    for alt in RELEASE_ALTS:
        sc = landing_scatter(L["body"], L["chute"], n=40, seed=7, wind_mean_range=(WIND - 1.5, WIND + 1.5), altitude_m=alt,
                             speed_m_s=0.8 * SIZES[name]["v_cruise"], timer_s=L["timer"], wind=Wind(0, 1.5, 2.0, 0.5), crush_stroke_m=STROKE, dt=0.02)
        around = np.abs(sc["drift_m"] - sc["drift_mean_m"])
        scatter_rows[(name, f"release {alt:.0f} m")] = {"aim upwind [m]": sc["drift_mean_m"], "scatter p95 [m]": float(np.percentile(around, 95)),
                                                        "worst opening g": sc["opening_g_max"], "worst touchdown g": sc["touchdown_g_max"],
                                                        "descent [m/s]": float(np.median(sc["descent_rate_m_s"])),
                                                        f"within {LANDING_CIRCLE_M:.0f} m": bool(np.percentile(around, 95) <= LANDING_CIRCLE_M)}
    ok_alts = [alt for alt in RELEASE_ALTS if scatter_rows[(name, f"release {alt:.0f} m")][f"within {LANDING_CIRCLE_M:.0f} m"]]
    L["release_alt"] = max(ok_alts) if ok_alts else min(RELEASE_ALTS)         # the highest release that still lands in the circle
    scatter[name] = dict(scatter_rows[(name, f"release {L['release_alt']:.0f} m")], release_alt_m=L["release_alt"])
    print(f"{name}: release at {L['release_alt']:.0f} m -> scatter p95 {scatter[name]['scatter p95 [m]']:.0f} m (circle {LANDING_CIRCLE_M:.0f} m)")
pd.DataFrame(scatter_rows).T

## 6b. Stresses in the whole construction (2 m)

The half wing in part 5 is clamped at a symmetry plane that does not exist in flight. Here the **whole 2 m airframe**
— wing, body, both fins and the pusher boss — is one solid of the same solid-equivalent foam, with two planar
**hardpoints** cut into the body (`hardpoint_depth`): a **belly skid** (the catapult shuttle, the landing skid and the
seat of the payload / battery / systems pack) and a **spine rail** (the parachute bridle). The pack and the motor are
lumped masses on their hardpoints. Every case is **self-balanced**: the air or attachment load one way, the inertia
of the airframe and of the lumped masses the other, so the support at the hardpoint only carries the residual.
Cases: the 2.5 g pull-up, the 8 g catapult launch, the **parachute opening** (its g from part 6), the **touchdown**
(the raw peak of the pad pulse as a static equivalent), the side gust on one fin together with the cruise thrust on
the boss, and the modes of the airframe on its skid.

In [ ]:
name = "2 m"; s2 = {k: v for k, v in SIZES[name].items() if k in PARAM_NAMES}; p = geo[name]["params"]
c0, b2, h, db = p["root_chord"], p["span"] / 2, p["body_height"], p["boss_diameter"]
HP = 6.0; zf = h / 2 - HP                                                            # the flats: skid at z = -zf, rail at z = +zf
whole = design.generate(part="aircraft", propulsion="pusher", hardpoint_depth=HP, **s2).export(ROOT / "fea" / "2m" / "aircraft", formats=("step",))
yf, cf, hf, tf, cant = p["fin_position"] * b2, p["fin_chord"], p["fin_height"], p["fin_thickness"], math.radians(p["fin_cant_deg"])
RW = [talos.SurfacesInBox("lower", (-1, -b2 - 1, -zf + 5, c0 + 1, b2 + 1, 0.5)),    # the lower wing skin: below z = 0, above the skid flat
      talos.SurfacesOnPlane("skid", "z", -zf, tol=0.5), talos.SurfacesOnPlane("rail", "z", zf, tol=0.5),
      talos.SurfacesInBox("fin_r", (-1, yf - 0.5 * tf - 2, -0.03 * c0, cf + 1, yf + hf * math.sin(cant) + tf + 2, hf + 5)),   # the +Y fin, both faces
      talos.SurfacesInBox("boss", (-db, -0.6 * db, -0.6 * db, 0.05 * c0, 0.6 * db, 0.6 * db))]
row = mass.loc[(name, "pusher / electric")]
W = float(row["take-off"]) * G; S_mm2 = geo[name]["S"] * 1e6
m_pack = float(row["payload"] + row["energy store"] + row["systems"] + row["parachute"]) * 1e-3       # tonnes, on the skid
m_motor = float(row["powerplant"]) * 1e-3                                                             # tonnes, on the boss
MASSES = [talos.PointMass("skid", m_pack), talos.PointMass("boss", m_motor)]
d2 = landing[name]["drop"]; g_open = d2.opening_g; g_td = crush_pulse(d2.touchdown_vertical_m_s, STROKE)[2]
v = SIZES[name]["v_cruise"]; S_fin = 0.5 * (cf + 0.55 * cf) * hf / 1e6                              # m^2, the fin's trapezium
F_fin = 0.5 * RHO * v * v * S_fin * 2.5 * (GUST / v)                                                 # N, the side gust (a = 2.5 /rad as above)
T_cruise = float(np.median(logs[(name, "pusher / electric")].thrust))
def lift(n): return talos.Pressure("lower", n * W / S_mm2)
CASES = {                                        # inertia (d'Alembert) loads: the body force is minus the acceleration
    "pull-up 2.5 g":     ([lift(N_PULL), talos.Acceleration(az=-N_PULL * 9810.0)], "skid"),
    "launch 8 g":        ([lift(1.0), talos.Acceleration(ax=-N_LAUNCH * 9810.0, az=-9810.0)], "skid"),
    "parachute opening": ([talos.Acceleration(az=-g_open * 9810.0)], "rail"),
    "touchdown":         ([talos.Acceleration(az=-g_td * 9810.0)], "skid"),
    "fin gust + thrust": ([lift(1.0), talos.Acceleration(az=-9810.0), talos.Force("fin_r", fy=-F_fin), talos.Force("boss", fx=T_cruise)], "skid"),
}
def whole_model(loads, support, tag):
    return talos.StructuralModel(whole.artifacts["step"], "mm-N-MPa", FOAM, RW, [talos.FixedSupport(support)], loads,
                                 talos.MeshSettings(element_size=30.0), name=tag, masses=MASSES)
def wdir(tag):
    d = ROOT / "fea" / "2m" / f"aircraft_{tag}"
    if not d.exists(): shutil.copytree(ROOT / "fea" / "2m" / "aircraft_mesh", d)
    return d
first = whole_model(*CASES["pull-up 2.5 g"], "pull_up")
wmesh = first.mesh(ROOT / "fea" / "2m" / "aircraft_mesh", progress=False); wmesh.raise_for_status()
print(f"whole 2 m airframe: {wmesh.metrics['n_elements']} elements, {wmesh.metrics['n_nodes']} nodes; pack {m_pack * 1e3:.1f} kg on the skid, motor {m_motor * 1e3:.1f} kg on the boss; "
      f"opening {g_open:.1f} g, touchdown {g_td:.0f} g, fin gust {F_fin:.0f} N, thrust {T_cruise:.0f} N")
whole_res, wrows = {}, {}
for tag, (loads, support) in CASES.items():
    r = whole_model(loads, support, tag.replace(" ", "_")).solve(wdir(tag.replace(" ", "_").replace(".", "")), progress=False); r.raise_for_status()
    whole_res[tag] = r; m = r.metrics; loc = m["max_von_mises_location"]
    wrows[tag] = {"support": support, "max von Mises [MPa]": m["max_von_mises"], "at x, y, z [mm]": tuple(round(c) for c in loc),
                  "max displacement [mm]": m["max_displacement"], "SF (yield)": FOAM.yield_strength / m["max_von_mises"]}
whole_modes = first.solve_modes(wdir("modal"), n_modes=8, progress=False); whole_modes.raise_for_status()
whole_table = pd.DataFrame(wrows).T
worst = whole_table["max von Mises [MPa]"].astype(float).idxmax()
print(f"worst case: {worst} -> {whole_table.loc[worst, 'max von Mises [MPa]']:.2f} MPa (SF {whole_table.loc[worst, 'SF (yield)']:.1f}); "
      f"airframe modes on the skid {[round(f, 1) for f in whole_modes.metrics['frequencies_hz'][:6]]} Hz")
whole_table

In [ ]:
tviz.show(tviz.plot_results(whole_res[worst], field="von_mises"))
tviz.show(tviz.plot_results(whole_res["pull-up 2.5 g"], field="U"))

**Reading it:** the pull-up and the gust load the wing as in part 5 (the same order of stress, now with the body
and the fins carrying their share); the launch, the opening and the touchdown are decided by the **lumped masses on
the hardpoints** — the stress concentrates at the edges of the skid and rail flats, which is where a real airframe
needs a hardpoint plate (a plywood or CFRP insert bonded into the foam). The solid-equivalent foam **underestimates the
skin stress** of a sandwich (the skin carries the bending): the coupon tests, then a shell-and-core model, are the next
step; the numbers here rank the cases and point at the hotspots.

## 7. Does the medicine survive?

Three loads reach the box, which sits on **isolators** (a mount at 25 Hz, 10 % damping — **assumed**):

1. **cruise vibration** — the propeller's unbalance at 1P (or the fan's), the piston engine's firing, the turbojet's
   broadband: the airframe's acceleration times the mount's transmissibility at that frequency;
2. the **opening shock** — the inflation force history through the mount (a shock response spectrum);
3. the **touchdown** — the crush pulse through the mount.

Each is compared with the limits at the top: vials (short pulses) and the vaccine (sustained shock, vibration).

In [ ]:
MOUNT_HZ, MOUNT_ZETA = 25.0, 0.10
def transmissibility(f_hz):
    r = f_hz / MOUNT_HZ; return math.sqrt((1 + (2 * MOUNT_ZETA * r) ** 2) / ((1 - r * r) ** 2 + (2 * MOUNT_ZETA * r) ** 2))
def assess(stroke_m):
  verdict = {}
  for (name, case), pp in PP.items():
    m_to = float(mass.loc[(name, case), "take-off"]); L = landing[name]; d = L["drop"]
    if "pusher" in case:                                           # 1P unbalance of the propeller (ISO G6.3) at the mission rpm
        prop = pp.propulsion.prop if case == "pusher / electric" else pp.prop
        rpm = noise[(name, case)]["rpm"]; f1 = rpm / 60
        F = boreas.unbalance_force(prop.rotor_mass_kg, rpm) * (3.0 if "petrol" in case else 1.0)     # petrol: firing pulses (assumed x3)
        a_air = F / (m_to * G); src = f"1P {f1:.0f} Hz"
    elif case == "jet / electric":
        f1 = 30000 / 60; F = boreas.unbalance_force(0.06 * (HW[name]["fan_d"] / 0.09) ** 2, 30000); a_air = F / (m_to * G); src = "fan 1P 500 Hz"
    else:
        f1 = 800.0; a_air = 0.5; src = "turbojet broadband ~0.5 g (assumed)"
    vib_g = transmissibility(f1) * a_air
    # opening shock: the base acceleration history around the opening, through the mount
    w = (d.t >= L["timer"] - 0.2) & (d.t <= L["timer"] + 4 * FILL)
    open_g = chronos.shock_at_mount(d.t[w] - d.t[w][0], (d.g[w] - 1.0) * G, MOUNT_HZ, MOUNT_ZETA) / G   # minus 1 g: the steady hang
    a_p, tau, g_raw = crush_pulse(d.touchdown_vertical_m_s, stroke_m); tt, aa = chronos.half_sine(a_p, tau, tail_s=0.2)
    td_g = chronos.shock_at_mount(tt, aa, MOUNT_HZ, MOUNT_ZETA) / G
    verdict[(name, case)] = {"cruise vibration [g rms]": vib_g / math.sqrt(2), "source": src, "opening shock at the box [g]": open_g,
                             "touchdown at the box [g]": td_g, "raw touchdown [g]": g_raw,
                             "vibration ok": vib_g / math.sqrt(2) <= G_VIB_RMS, "opening ok": open_g <= G_VACCINE, "touchdown ok": td_g <= G_VIALS}
  verdict = pd.DataFrame(verdict).T
  verdict["medicine intact"] = verdict[["vibration ok", "opening ok", "touchdown ok"]].all(axis=1)
  return verdict
verdict = assess(STROKE)
print(f"with the {STROKE * 1000:.0f} mm pad: {int(verdict['medicine intact'].sum())} of {len(verdict)} cases keep the medicine intact")
verdict

In [ ]:
# the levers: pad stroke and mount frequency against the touchdown at the box, for the heavier landing
name = "3 m"; d = landing[name]["drop"]
strokes = [0.02, 0.04, 0.06, 0.08, 0.12]; mounts = [10.0, 25.0, 50.0]
lever = pd.DataFrame({f"mount {mh:.0f} Hz": {f"pad {s * 1000:.0f} mm": chronos.shock_at_mount(*chronos.half_sine(*crush_pulse(d.touchdown_vertical_m_s, s)[:2], tail_s=0.2), mh, MOUNT_ZETA) / G
                                             for s in strokes} for mh in mounts})
lever.index.name = f"{name}, touchdown {d.touchdown_vertical_m_s:.1f} m/s -> g at the box"
print(f"limit: vials {G_VIALS:.0f} g; a softer mount and a longer pad both help, the pad most")
lever.round(1)

In [ ]:
# the fix: a longer crush pad (the lightest change: foam under the bay), the same mount
STROKE_FIX = 0.10
verdict_fix = assess(STROKE_FIX)
print(f"with a {STROKE_FIX * 1000:.0f} mm pad: {int(verdict_fix['medicine intact'].sum())} of {len(verdict_fix)} cases keep the medicine intact; "
      f"touchdown at the box {verdict_fix['touchdown at the box [g]'].max():.0f} g (limit {G_VIALS:.0f} g)")
verdict_fix[["opening shock at the box [g]", "touchdown at the box [g]", "raw touchdown [g]", "medicine intact"]]

**Reading the verdict:** the cruise vibration is small at the box for every case — the 25 Hz mount isolates the
propeller and fan lines, and the turbojet's broadband is an assumption to measure. The opening shock depends on the
timer: the sweep in part 6 picked the gentlest opening that still leaves 40 m of altitude. The touchdown is what
the pad decides: without one, a 5 m/s landing is hundreds of g at the box; with 60 mm of crush, tens. The design
levers are the timer, the canopy area (descent rate), the pad stroke and the mount — all explicit inputs above; the release
altitude sets the landing scatter (the wind uncertainty works for the whole descent).

## 8. Export, and the descent as a movie

In [ ]:
summary = {
    "requirements": {"radius_km": RADIUS_KM, "wind_m_s": WIND, "landing_circle_m": LANDING_CIRCLE_M, "g_vials": G_VIALS, "g_vaccine": G_VACCINE, "g_vib_rms": G_VIB_RMS},
    "sizes": {n: {"planform_m2": g["S"], "AR": g["AR"], "sweep_deg": g["sweep_deg"], "flight": flight[n], "aero": {"Kp": aero[n]["Kp"], "k": aero[n]["k"], "cd0": CD0}} for n, g in geo.items()},
    "mass_budget": {f"{n} | {c}": {k: float(v) for k, v in row.items()} for (n, c), row in mass.iterrows()},
    "missions": {f"{n} | {c}": logs[(n, c)].summary() for (n, c) in logs},
    "comparison": {f"{n} | {c}": {k: (float(v) if not isinstance(v, (bool, str)) else v) for k, v in row.items()} for (n, c), row in comparison.iterrows()},
    "structure": {n: {"pull_up_max_MPa": s["pull"].metrics["max_von_mises"], "launch_max_MPa": s["launch"].metrics["max_von_mises"], "fin_gust_max_MPa": s["fin"].metrics["max_von_mises"],
                      "modes_hz": s["modes"].metrics["frequencies_hz"], "material": FOAM.name} for n, s in structure.items()},
    "whole_construction_2m": {"cases": {tag: {"support": r["support"], "max_von_mises_MPa": float(r["max von Mises [MPa]"]), "location_mm": list(r["at x, y, z [mm]"]),
                                              "max_displacement_mm": float(r["max displacement [mm]"]), "safety_factor": float(r["SF (yield)"])} for tag, r in wrows.items()},
                              "modes_on_skid_hz": whole_modes.metrics["frequencies_hz"], "n_elements": wmesh.metrics["n_elements"], "hardpoint_depth_mm": HP,
                              "pack_kg": m_pack * 1e3, "motor_kg": m_motor * 1e3, "opening_g": g_open, "touchdown_g": g_td, "fin_gust_N": F_fin, "thrust_N": T_cruise},
    "landing": {n: {"canopy_m2": L["chute"].area_m2, "timer_s": L["timer"], **L["drop"].peaks(), **scatter[n]} for n, L in landing.items()},
    "verdict": {f"pad {STROKE * 1000:.0f} mm": {f"{n} | {c}": {k: (float(v) if not isinstance(v, (bool, str)) else v) for k, v in row.items()} for (n, c), row in verdict.iterrows()},
                f"pad {STROKE_FIX * 1000:.0f} mm": {f"{n} | {c}": {k: (float(v) if not isinstance(v, (bool, str)) else v) for k, v in row.items()} for (n, c), row in verdict_fix.iterrows()}},
    "cfd": {"clean": {n: (r.metrics if r is not None and r.ok else "not run") for n, (case, r) in cfd_clean.items()},
            "propulsion": {f"{n} {p}": (r.metrics if r is not None and r.ok else "not run") for (n, p), (case, r) in cfd_prop.items()}},
}
(ROOT / "delta_wing_delivery.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", ROOT / "delta_wing_delivery.json")

In [ ]:
# the 2 m landing: motor cut, canopy, descent in the gusty wind, touchdown (matplotlib frames -> MP4 with the Vegeta watermark)
from IPython.display import Video
name = "2 m"; d = landing[name]["drop"]; L = landing[name]
fps, seconds = 12, 8.0
idx = np.linspace(0, len(d.t) - 1, int(fps * seconds)).astype(int)
frames = []
for k in idx:
    fig, ax = plt.subplots(figsize=(9.6, 5.4), dpi=100)
    ax.plot(d.x[:k + 1], d.z[:k + 1], color="#1565c0", lw=1.5)
    ax.plot(d.x[k], d.z[k], "v", color="#37474f", ms=12)
    if d.phase[k] >= 1:
        frac = 1.0 if d.phase[k] == 2 else ((d.t[k] - L["timer"]) / FILL) ** 2
        r_c = 0.5 * math.sqrt(4 * L["chute"].area_m2 / math.pi) * math.sqrt(frac) * 1.0
        ax.add_patch(plt.Circle((d.x[k], d.z[k] + 6), r_c, fill=True, color="#ef6c00", alpha=0.5))
    ax.set(xlim=(-10, max(50, d.x[-1] + 20)), ylim=(0, CRUISE_ALT_M + 20), xlabel="drift with the wind [m]", ylabel="altitude [m]")
    ax.set_title(f"{name} delta wing, timed parachute: t = {d.t[k]:5.1f} s   {['falling', 'canopy opening', 'descending'][d.phase[k]]}   {d.g[k]:.1f} g   "
                 f"descent {d.descent_speed[k]:.1f} m/s (speed {d.speed[k]:.1f} m/s)")
    ax.grid(alpha=0.3)
    ins = ax.inset_axes([0.62, 0.55, 0.35, 0.3])                                      # the speed trace up to this frame
    ins.plot(d.t, d.descent_speed, color="#b0bec5", lw=0.8); ins.plot(d.t[:k + 1], d.descent_speed[:k + 1], color="#c62828", lw=1.5)
    ins.plot(d.t[k], d.descent_speed[k], "o", color="#c62828", ms=4)
    ins.set(xlim=(0, d.t[-1]), ylim=(min(0, d.descent_speed.min()) - 1, d.descent_speed.max() + 2)); ins.set_title("descent speed [m/s]", fontsize=8)
    ins.tick_params(labelsize=7); ins.grid(alpha=0.3)
    fig.tight_layout(); fig.canvas.draw()
    frames.append(np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()); plt.close(fig)
video = aviz.write_video(frames, MOVIES / "landing_2m.mp4", fps=fps)
display(Video(str(video), embed=False, width=720))

**What is on disk:** `_runs/delta_wing/delta_wing_delivery.json` (every input, mass budget, mission, range, structure,
landing and verdict), the prepared CFD cases (`cfd_clean/`, `cfd_propulsion/`), the FEA cases (`fea/`), and the
movie in `output/16_delta_wing_delivery/`. A fully resolved propeller or engine, a free-surface anything, and the
coupon tests behind the assumed materials and g limits are the next steps — the notebook only makes the chain
visible and repeatable; the decisions are yours.